# Глава 7. Логистическая регрессия

Код к главе 7 книги 1: почему не годится линейная регрессия, отступ и функции потерь,
сигмоида и шансы, максимум правдоподобия, градиентный спуск для логистической потери,
регуляризация и логистическая регрессия в scikit-learn (листинг 7.1<!-- lst:Логистическая регрессия -->) на данных о модулях
программы.

[![Открыть в Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/markpolyak/ml-textbook-code/blob/main/book1/ch07_logreg.ipynb)

In [ ]:
# В Colab ставим пакет mlbook с данными книги (локально: pip install -e . в корне репозитория)
import sys

if "google.colab" in sys.modules:
    %pip install -q git+https://github.com/markpolyak/ml-textbook-code

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from mlbook import modules

## Почему не подходит линейная регрессия

Восемь опухолей с метками $\pm 1$: прямая МНК пересекает нуль в точке 3 см. Одна далёкая
злокачественная опухоль (14 см) сдвигает порог до 3,53 см, и объект 3,5 см классифицируется
неверно (рис. 7.1<!-- fig:linreg-class -->).

In [ ]:
x = np.array([1, 1.5, 2, 2.5, 3.5, 4, 4.5, 5])
y = np.array([-1, -1, -1, -1, 1, 1, 1, 1.0])


def threshold(x, y):
    t0, t1 = np.polyfit(x, y, 1)[::-1]
    return -t0 / t1


print(f"порог: {threshold(x, y):.2f} → {threshold(np.r_[x, 14], np.r_[y, 1]):.2f}")

## Линейный классификатор и отступ

$\boldsymbol{\theta} = (-3;\ 1;\ 1)$: разделяющая функция $h = -3 + x_1 + x_2$, граница —
прямая $x_1 + x_2 = 3$. Отступ $M = y\,h$ положителен при верном ответе.

In [ ]:
theta = np.array([-3, 1, 1])
Xo = np.array([[1, 1, 1], [1, 2, 2]])       # объекты (1; 1) и (2; 2) с f0 = 1
h = Xo @ theta
print("h =", h, " классы:", np.sign(h), " отступы при y = (−1, +1):", np.array([-1, 1]) * h)

Пороговая функция потерь и её гладкие верхние оценки (рис. 7.3<!-- fig:losses -->).

In [ ]:
M = np.linspace(-2.5, 3, 500)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(M, (M < 0).astype(float), color="black", linewidth=2, label="пороговая $[M < 0]$")
ax.plot(M, np.log2(1 + np.exp(-M)), label=r"логистическая $\log_2(1 + e^{-M})$")
ax.plot(M, np.maximum(0, 1 - M), label=r"кусочно-линейная $\max(0, 1 - M)$")
ax.plot(M, np.exp(-M), label="экспоненциальная $e^{-M}$")
ax.set_ylim(-0.1, 4)
ax.set_xlabel("отступ $M$")
ax.grid(True)
ax.legend()
plt.show()

## Сигмоида, вероятность и шансы

$\sigma(z) = 1/(1 + e^{-z})$; свойства $\sigma(-z) = 1 - \sigma(z)$ и
$\sigma' = \sigma(1 - \sigma)$ проверим численно.

In [ ]:
sigma = lambda z: 1 / (1 + np.exp(-z))
z = np.linspace(-6, 6, 13)
num_deriv = (sigma(z + 1e-6) - sigma(z - 1e-6)) / 2e-6
print(np.allclose(sigma(-z), 1 - sigma(z)), np.allclose(num_deriv, sigma(z) * (1 - sigma(z))))

In [ ]:
zs = np.linspace(-6, 6, 400)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(zs, sigma(zs), label=r"$\sigma(z)$")
ax.plot(zs, sigma(zs) * (1 - sigma(zs)), "--", label=r"$\sigma'(z)$")
ax.axhline(0.5, color="0.6", linewidth=0.6)
ax.set_xlabel("$z$")
ax.grid(True)
ax.legend()
plt.show()

Шансы $p/(1 - p) = e^h$. Объект $(1;\ 1)$: $h = -1$, вероятность класса 1 около 0,27,
шансы $e^{-1} \approx 0{,}37$. Увеличение $x_1$ на единицу умножает шансы на $e$.

In [ ]:
for obj in ([1, 1], [2, 1]):
    hh = theta @ np.r_[1, obj]
    p = sigma(hh)
    print(f"объект {obj}: h = {hh}, p = {p:.2f}, шансы p/(1 − p) = {p / (1 - p):.2f}")

## Максимум правдоподобия

Три объекта $x = 1, 2, 3$ с метками $-1, +1, +1$. Правдоподобие — произведение вероятностей
правильных ответов $\sigma(M_i)$; минус его логарифм — сумма логистических потерь.

In [ ]:
x3, y3 = np.array([1, 2, 3]), np.array([-1, 1, 1])
for th in ([-3, 2], [-1.5, 1], [-6, 4]):
    Mi = y3 * (th[0] + th[1] * x3)
    L = np.prod(sigma(Mi))
    print(f"θ = {th}: отступы {Mi}, L = {L:.3f}, −ln L = {np.sum(np.log1p(np.exp(-Mi))):.3f}")

Классы в этой выборке разделимы, и без регуляризации веса растут бесконечно: чем слабее
штраф (больше `C`), тем больше веса.

In [ ]:
from sklearn.linear_model import LogisticRegression

for C in (1, 100, 1e4, 1e6):
    m = LogisticRegression(C=C, max_iter=10_000).fit(x3.reshape(-1, 1), y3)
    print(f"C = {C:g}: θ0 = {m.intercept_[0]:.2f}, θ1 = {m.coef_[0, 0]:.2f}")

## Градиент и градиентный спуск

$\nabla Q = \frac{1}{\ell}\mathbf{F}^{\mathrm{T}}(\mathbf{p} - \tilde{\mathbf{y}})$ с метками
$\tilde y \in \{0, 1\}$. Проверим формулу численно и обучим модель спуском на данных о
модулях программы.

In [ ]:
df = modules.load()
X = df[modules.FEATURES].to_numpy(dtype=float)
yt = df["ошибка"].to_numpy()                          # метки 0 и 1
Z = (X - X.mean(0)) / X.std(0)
F = np.column_stack([np.ones(len(Z)), Z])


def Q(th):                                             # средняя бинарная кросс-энтропия
    p = sigma(F @ th)
    return -np.mean(yt * np.log(p) + (1 - yt) * np.log(1 - p))


def grad(th):
    return F.T @ (sigma(F @ th) - yt) / len(yt)


th = np.array([0.1, -0.2, 0.3, 0.1, -0.1])
num = np.array([(Q(th + e) - Q(th - e)) / 2e-6 for e in 1e-6 * np.eye(5)])
print("формула совпадает с численной производной:", np.allclose(num, grad(th), atol=1e-8))

In [ ]:
th = np.zeros(5)
for t in range(2000):
    th = th - 0.5 * grad(th)
sk = LogisticRegression(penalty=None).fit(Z, yt)
print("спуск:        ", th.round(3), f" Q = {Q(th):.4f}")
print("scikit-learn: ", np.r_[sk.intercept_, sk.coef_[0]].round(3))

## Регуляризация

Параметр `C` — обратный коэффициент регуляризации: чем он меньше, тем сильнее веса
стянуты к нулю.

In [ ]:
for C in (0.001, 0.01, 0.1, 1, 100):
    m = LogisticRegression(C=C).fit(Z, yt)
    print(f"C = {C:6}: веса {m.coef_[0].round(3)}")

## Несколько классов: softmax

При $K = 2$ softmax совпадает с сигмоидой от разности $h_1 - h_2$ (задача 3).

In [ ]:
def softmax(h):
    e = np.exp(h - h.max())
    return e / e.sum()


h12 = np.array([0.7, -0.4])
print(softmax(h12)[0], sigma(h12[0] - h12[1]))

## Логистическая регрессия в scikit-learn

Листинг 7.1<!-- lst:Логистическая регрессия -->. Разобьём выборку модулей на обучающую и тестовую части; стратификация
сохраняет долю модулей с ошибками в обеих частях.

In [ ]:
from sklearn.model_selection import train_test_split

X = df[modules.FEATURES]
y = df["ошибка"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=0)

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

model = make_pipeline(StandardScaler(),
                      LogisticRegression(C=1.0))
model.fit(X_train, y_train)
proba = model.predict_proba(X_test)[:, 1]  # P(класс 1)
y_pred = model.predict(X_test)             # порог 0,5
odds = np.exp(model[-1].coef_)             # множители шансов

Во сколько раз меняются шансы ошибки при увеличении признака на одно стандартное
отклонение (признаки стандартизованы). Наличие тестов снижает шансы, размер модуля и
частота изменений — повышают.

In [ ]:
for name, o in zip(modules.FEATURES, odds[0]):
    print(f"{name:10} ×{o:.2f}")
print("первые пять вероятностей:", proba[:5].round(2))
print("доля верных ответов на тесте:", round(np.mean(y_pred == y_test), 3))
print("то же у правила «ошибок нет»:", round(np.mean(y_test == 0), 3))

Доля верных ответов почти такая же, как у бесполезного правила «ошибок нет в любом модуле»:
модулей с ошибками меньше четверти. Почему эта метрика обманчива и чем её заменить —
в гл. 8.

## Задачи

**1.** $\boldsymbol{\theta} = (-3;\ 1;\ 1)$, объекты $(3;\ 1)$ и $(0;\ 1)$.

In [ ]:
for obj in ([3, 1], [0, 1]):
    hh = theta @ np.r_[1, obj]
    print(f"объект {obj}: h = {hh}, p = {sigma(hh):.2f}, шансы {np.exp(hh):.2f}")

**2.** Бинарная кросс-энтропия для $\tilde y = 1$ при $p = 0{,}9;\ 0{,}5;\ 0{,}1$.

In [ ]:
from sklearn.metrics import log_loss

for p in (0.9, 0.5, 0.1):
    print(f"p = {p}: −ln p = {-np.log(p):.3f}, log_loss = {log_loss([1], [p], labels=[0, 1]):.3f}")

**4.** Шаг спуска для потери на объекте $x = 2$, $\tilde y = 1$ из $(0;\ 0)$ при $\eta = 0{,}5$.

In [ ]:
xi, th = np.array([1.0, 2.0]), np.zeros(2)
g = (sigma(th @ xi) - 1) * xi
print("градиент:", g, " θ =", th - 0.5 * g)